In [2]:
import requests
import json
import os
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
URL = "https://anilist.co/"
headers = {
    "User-Agent": "Anime Scrapper"
}

In [4]:
response = requests.get(URL, headers=headers)

In [5]:
print(response)

<Response [200]>


In [6]:
print(response.content)

b'<!DOCTYPE html><html lang=en><head><title>AniList</title><meta charset=utf-8><meta http-equiv=x-ua-compatible content="ie=edge"><meta http-equiv=Content-Security-Policy content=block-all-mixed-content><meta name=viewport content="width=device-width,initial-scale=1,maximum-scale=1,minimum-scale=1,user-scalable=no"><meta property=og:site_name content=AniList><meta name=twitter:site content=@AniListco><script>window.al_token = "R4bXSASTpO6wLPq7UVDoAhEBCShJ47knsDEZCR6J";</script><link href="//fonts.googleapis.com/css?family=Roboto:300,400,500,700" rel=stylesheet><link rel=preload as=style href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800"><link href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800" rel=stylesheet><link rel=icon type=image/png sizes=32x32 href=/img/icons/favicon-32x32.png><link rel=icon type=image/png sizes=16x16 href=/img/icons/favicon-16x16.png><link rel=manifest href=/manifest.json><meta name=theme-color content=#2b2d42><meta name=

In [7]:
#Create a Beautiful Soup object
soup = BeautifulSoup(response.text, 'lxml')

In [8]:
print(soup)

<!DOCTYPE html>
<html lang="en"><head><title>AniList</title><meta charset="utf-8"/><meta content="ie=edge" http-equiv="x-ua-compatible"/><meta content="block-all-mixed-content" http-equiv="Content-Security-Policy"/><meta content="width=device-width,initial-scale=1,maximum-scale=1,minimum-scale=1,user-scalable=no" name="viewport"/><meta content="AniList" property="og:site_name"/><meta content="@AniListco" name="twitter:site"/><script>window.al_token = "R4bXSASTpO6wLPq7UVDoAhEBCShJ47knsDEZCR6J";</script><link href="//fonts.googleapis.com/css?family=Roboto:300,400,500,700" rel="stylesheet"/><link as="style" href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800" rel="preload"/><link href="https://fonts.googleapis.com/css?family=Overpass:400,600,700,800" rel="stylesheet"/><link href="/img/icons/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/><link href="/img/icons/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/><link href="/manifest.json" rel="ma

In [9]:
manhwas = soup.find_all("div", class_="feature-cards")

In [10]:
print(len(manhwas))

0


In [11]:
manhwas

[]

In [12]:
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

In [13]:
driver = webdriver.Chrome()
driver.get("https://anilist.co/")

# Wait for page to load
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "feature-cards")))

# Now you can scrape
soup = BeautifulSoup(driver.page_source, 'html.parser')
manhwas = soup.find_all("div", class_="feature-cards") 

print(f"Found {len(manhwas)} feature cards")

# Or use Selenium directly
elements = driver.find_elements(By.CLASS_NAME, "feature-cards")
for elem in elements:
    print(elem.text[:100]) 


Found 1 feature cards
Discover your obsessions
What are your highest rated genres or most watched voice actors? Follow you


In [14]:
!pip install webdriver-manager


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

def scrape_anilist_manhwa():
    # Setup driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    
    # Target the public landing page instead of /home
    driver.get("https://anilist.co/") 
    
    wait = WebDriverWait(driver, 15)
    try:
        # 1. Wait for the actual media card container elements to load
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "cover")))
        
        # Scroll to ensure everything triggers/renders
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 4);")
        time.sleep(2)
        
        # 2. Update to the correct class structure on the landing page
        # AniList items typically sit inside cards with 'title' elements inside them
        cards = driver.find_elements(By.CLASS_NAME, "media-card")
        manhwa_data = []
        
        for card in cards[:20]:
            try:
                # Extract title text
                title_elem = card.find_element(By.CLASS_NAME, "title")
                title = title_elem.text.strip()
                
                # If titles are empty (sometimes happens before hover/interaction), 
                # grab the text directly or content attributes
                if not title:
                    title = card.text.split('\n')[0] 
                
                manhwa_data.append({
                    'Title': title,
                    'Type': 'Manga/Anime' # The landing page mixes both
                })
            except Exception as e:
                continue
                
        return pd.DataFrame(manhwa_data)
        
    finally:
        driver.quit()


In [16]:

# Run the scraper
df = scrape_anilist_manhwa()
print(df.head())

                                               Title         Type
0   Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
1  Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
2                    Otaku ni Yasashii Gal wa Inai!?  Manga/Anime
3                          Tongari Boushi no Atelier  Manga/Anime
4                                          ONE PIECE  Manga/Anime


In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument('--headless')  # Run without opening browser window
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.get("https://anilist.co/")
print("Page loaded in headless mode")
driver.quit()

Page loaded in headless mode


In [18]:
print(df)
print(f"Total records: {len(df)}")

                                                Title         Type
0    Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
1   Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
2                     Otaku ni Yasashii Gal wa Inai!?  Manga/Anime
3                           Tongari Boushi no Atelier  Manga/Anime
4                                           ONE PIECE  Manga/Anime
5   Class de 2-banme ni Kawaii Onnanoko to Tomodac...  Manga/Anime
6                           Tongari Boushi no Atelier  Manga/Anime
7    Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
8                                      Yomi no Tsugai  Manga/Anime
9           Tensei Shitara Slime Datta Ken 4th Season  Manga/Anime
10  Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
11               Tsue to Tsurugi no Wistoria Season 2  Manga/Anime
12       Mushoku Tensei III: Isekai Ittara Honki Dasu  Manga/Anime
13                                     Youjo Senki II  Manga/A

In [19]:
df.duplicated().sum()

np.int64(3)

In [20]:
df = df.drop_duplicates()

In [21]:
print(df)
print(f"Total unique records: {len(df)}")

                                                Title         Type
0    Re:Zero kara Hajimeru Isekai Seikatsu 4th Season  Manga/Anime
1   Youkoso Jitsuryoku Shijou Shugi no Kyoushitsu ...  Manga/Anime
2                     Otaku ni Yasashii Gal wa Inai!?  Manga/Anime
3                           Tongari Boushi no Atelier  Manga/Anime
4                                           ONE PIECE  Manga/Anime
5   Class de 2-banme ni Kawaii Onnanoko to Tomodac...  Manga/Anime
8                                      Yomi no Tsugai  Manga/Anime
9           Tensei Shitara Slime Datta Ken 4th Season  Manga/Anime
11               Tsue to Tsurugi no Wistoria Season 2  Manga/Anime
12       Mushoku Tensei III: Isekai Ittara Honki Dasu  Manga/Anime
13                                     Youjo Senki II  Manga/Anime
14                    Super no Ura de Yani Suu Futari  Manga/Anime
15             BLEACH: Sennen Kessen-hen - Kashin-tan  Manga/Anime
16    Mahou Shoujo Madoka☆Magica: Walpurgis no Kaiten  Manga/A